# URL-only quishing experiment

This notebook runs the shared CLI for the benign-only GRU VAE, Floors A and B,
and the supervised GRU. See the [README](../README.md) for the dataset, protocol,
and output definitions. Run from a checkout of this branch with the project
dependencies installed and the two QR URL CSVs available.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

root = Path.cwd()
while root != root.parent and not (root / "src" / "run_url.py").is_file():
    root = root.parent
if not (root / "src" / "run_url.py").is_file():
    raise FileNotFoundError("Open this notebook from the URL experiment checkout.")

benign_csv = root / "data" / "raw" / "benign_urls.csv"
malicious_csv = root / "data" / "raw" / "malicious_urls.csv"
output = root / "results" / "url_experiment"

command = [
    sys.executable, "-m", "src.run_url",
    "--benign-csv", str(benign_csv),
    "--malicious-csv", str(malicious_csv),
    "--output", str(output),
]

Edit the CSV paths before running. The output
folder must be fresh. Append `--prepare-only` for a separate split and floor
audit. The default experiment trains on CUDA when available.

In [ ]:
subprocess.run(command, cwd=root, check=True);

In [ ]:
import pandas as pd

evaluation = output / "evaluation.json"
if evaluation.is_file():
    report = json.loads(evaluation.read_text())
    display(pd.DataFrame({
        model: {metric: value["estimate"] for metric, value in metrics.items()}
        for model, metrics in report["overall"]["metrics"].items()
    }).T)
    display(report["success_criteria"])
else:
    audit = json.loads((output / "split_audit.json").read_text())
    display(audit["splits"])